# 05 — Test pédales vJoy

Contrôle facultatif des axes Y (accélérateur) et Z (frein) utilisés par le plugin original. **Ces commandes agissent sur le simulateur** : pas d’action sans validation manuelle.

In [10]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaEnv').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvre le depot IA_projet_Ouarti_ds_github dans VS Code : son dossier assetto_corsa_gym doit exister.')
PROJECT=REPO/'Code'
if not PROJECT.is_dir():
    raise FileNotFoundError('Dossier Code absent : extraire Code a la racine du depot.')
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
print('Depot :',REPO)
print('Notebooks :',PROJECT)
print('Python :',sys.version.split()[0],sys.executable)


Depot : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github
Notebooks : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code
Python : 3.9.13 c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe


In [11]:
import ctypes,time
from pathlib import Path
DLL_PATH=Path(r'C:\Program Files\vJoy\x64\vJoyInterface.dll')
DEVICE_ID=1
HID_USAGE_X=0x30

def get_vjoy():
    if not DLL_PATH.exists():raise FileNotFoundError(str(DLL_PATH)+' : installer vJoy et configurer device 1')
    dll=ctypes.CDLL(str(DLL_PATH))
    dll.GetVJDStatus.argtypes=[ctypes.c_uint];dll.GetVJDStatus.restype=ctypes.c_int
    dll.AcquireVJD.argtypes=[ctypes.c_uint];dll.AcquireVJD.restype=ctypes.c_bool
    dll.SetAxis.argtypes=[ctypes.c_long,ctypes.c_uint,ctypes.c_uint];dll.SetAxis.restype=ctypes.c_bool
    dll.RelinquishVJD.argtypes=[ctypes.c_uint];dll.RelinquishVJD.restype=None
    return dll


In [12]:
# # Valeur neutre d'apres les controles du plugin: X=16384 ; Y=0 ; Z=0.
# # Vérifier les axes dans vJoy Monitor / réglages d'Assetto Corsa avant d'activer.
# EXECUTER_TEST_PEDALES=True
# AXE_Y=0x31
# AXE_Z=0x32
# if not EXECUTER_TEST_PEDALES:
#     print('Aucune commande envoyee. Active EXECUTER_TEST_PEDALES seulement pour tester a l arret.')
# else:
#     dll=get_vjoy()
#     if dll.GetVJDStatus(DEVICE_ID)!=1 or not dll.AcquireVJD(DEVICE_ID):
#         raise RuntimeError('vJoy indisponible.')
#     try:
#         for axe,valeur in [(AXE_Y,8000),(AXE_Y,0),(AXE_Z,8000),(AXE_Z,0)]:
#             if not dll.SetAxis(valeur,DEVICE_ID,axe):raise RuntimeError('Echec SetAxis')
#             time.sleep(.6)
#     finally:
#         for axe,valeur in [(HID_USAGE_X,16384),(AXE_Y,0),(AXE_Z,0)]:
#             dll.SetAxis(valeur,DEVICE_ID,axe)
#         dll.RelinquishVJD(DEVICE_ID)


import time

# Configuration vJoy
DEVICE_ID = 1

AXE_X = 0x30  # Volant
AXE_Y = 0x31  # Accélérateur
AXE_Z = 0x32  # Frein

MIN = 0
MAX = 32767
CENTRE = 16384

# Paramètres du test
DUREE_ACCELERATION = 3.0
DUREE_DECELERATION = 3.0
DUREE_FREINAGE = 1.0
ETAPES = 30

EXECUTER_TEST_PEDALES = True

if not EXECUTER_TEST_PEDALES:
    print("Test désactivé.")
else:
    dll = get_vjoy()

    if dll.GetVJDStatus(DEVICE_ID) != 1:
        raise RuntimeError("vJoy Device 1 indisponible")

    if not dll.AcquireVJD(DEVICE_ID):
        raise RuntimeError("Impossible d'acquérir vJoy")

    def envoyer(axe, valeur):
        valeur = max(MIN, min(MAX, int(valeur)))
        if not dll.SetAxis(valeur, DEVICE_ID, axe):
            raise RuntimeError(f"Erreur sur l'axe {axe}")

    try:
        # Initialisation
        envoyer(AXE_X, CENTRE)
        envoyer(AXE_Y, MIN)
        envoyer(AXE_Z, MIN)

        # 1. Accélération progressive : 0 -> 100 %
        print("1. Accélération progressive")

        for i in range(ETAPES + 1):
            pourcentage = i / ETAPES
            valeur = round(MAX * pourcentage)

            envoyer(AXE_Y, valeur)

            print(f"\rAccélérateur : {pourcentage*100:.0f} %", end="")
            time.sleep(DUREE_ACCELERATION / ETAPES)

        print("\nAccélérateur à 100 %")

        # 2. Freinage pendant 1 seconde
        print("\n2. Freinage pendant 1 seconde")

        envoyer(AXE_Z, MAX)
        time.sleep(DUREE_FREINAGE)

        envoyer(AXE_Z, MIN)
        print("Frein relâché")

        # 3. Décélération progressive : 100 -> 0 %
        print("\n3. Décélération progressive")

        for i in range(ETAPES + 1):
            pourcentage = 1 - i / ETAPES
            valeur = round(MAX * pourcentage)

            envoyer(AXE_Y, valeur)

            print(f"\rAccélérateur : {pourcentage*100:.0f} %", end="")
            time.sleep(DUREE_DECELERATION / ETAPES)

        print("\nAccélérateur à 0 %")

        # 4. Freinage maximal
        print("\n4. Freinage maximal")

        envoyer(AXE_Y, MIN)
        envoyer(AXE_Z, MAX)

        print("Frein : 100 %")
        time.sleep(DUREE_FREINAGE)

        # 5. Relâchement des commandes
        envoyer(AXE_Z, MIN)
        print("\nTest terminé : pédales relâchées")

    finally:
        # Sécurité : remise à zéro même en cas d'erreur
        try:
            dll.SetAxis(CENTRE, DEVICE_ID, AXE_X)
            dll.SetAxis(MIN, DEVICE_ID, AXE_Y)
            dll.SetAxis(MIN, DEVICE_ID, AXE_Z)
        finally:
            dll.RelinquishVJD(DEVICE_ID)

        print("vJoy libéré.")



1. Accélération progressive
Accélérateur : 100 %
Accélérateur à 100 %

2. Freinage pendant 1 seconde
Frein relâché

3. Décélération progressive
Accélérateur : 0 %%%
Accélérateur à 0 %

4. Freinage maximal
Frein : 100 %

Test terminé : pédales relâchées
vJoy libéré.
